In [1]:
import lenstronomy
from astropy.io import fits
import numpy as np
import scipy as sp
import pickle 
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import minimize
from astropy.wcs import WCS
import pyswarms as ps
from os import listdir
from os.path import isfile, join
from scipy.optimize import linear_sum_assignment
from lenstronomy.Util import param_util
import os, emcee
import copy, math
from tqdm import tqdm
from pathlib import Path
import corner, time
from IPython.display import Image as imp
import astropy.units as u
from astropy.cosmology import FlatLambdaCDM

from vorbin.voronoi_2d_binning import voronoi_2d_binning

from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.Workflow.fitting_sequence import FittingSequence
from lenstronomy.LensModel.Solver.lens_equation_solver import LensEquationSolver
from lenstronomy.Data.psf import PSF
from lenstronomy.LightModel.Profiles.sersic import Sersic
from lenstronomy.Plots.model_plot import ModelPlot
from multiprocessing import Pool

import sys

sys.path.append("../src")


from iful.util import *
from iful.image_set import *
from iful.flat_modeling import *
from iful.iful_modeling import *

In [2]:
while not os.path.exists("s4_models/flat_chains.pickle"):
    time.sleep(60)

In [3]:
with open("s4_models/imset4.pickle", "rb") as handle:
    imset4 = pickle.load(handle)

with open("s4_models/flatmodel4.pickle", "rb") as handle:
    flatmodel4 = pickle.load(handle)

iful_pso_results_filename = "s4_models/iful_pso_results_params.pickle"
if os.path.isfile(iful_pso_results_filename):
    with open(iful_pso_results_filename, "rb") as handle:
        previous_results = pickle.load(handle)
else:
    previous_results = {}

## Setting up ifulmodel object

In [4]:
threadCount = 40

c = 299792
d_s = FlatLambdaCDM(H0=70, Om0=0.3).angular_diameter_distance(imset4.zs).to(u.kpc).value

In [ ]:
# imageset, flatmodel, iful_profiles, sourceplane_size, num_bins, num_rsersics, spectral_res, constant_val=0.):
iful_profiles = ["ARCTAN", "CONSTANT_FITTED_BH", "VORONOI"]
ifulmodel4 = IFULModel(
    imset4,
    flatmodel4,
    iful_profiles,
    sourceplane_size=100,
    num_bins=50,
    num_rsersics=3,
    spectral_res=3500,
    equal_weight_voronoi=False,
    d_s=d_s,
)

## Setting initial parameters

In [ ]:
lens_model_initparams = list(
    ifulmodel4.init_fitting_seq.param_class.kwargs2args(**flatmodel4.init_pso_fit)
)
# v_pa, v_a, v_b, v_c
v_los_initparams = [75, 150, 7, imset4.zs * c]
# const_val, lg_bh_mass
v_disp_initparams = [50, 8.0]
# bin_vals
flx_initparams = []# [flatmodel4.init_pso_fit['kwargs_source'][0]['R_sersic']] 

init_params = (
    lens_model_initparams + v_los_initparams + v_disp_initparams + flx_initparams
)
print(len(init_params))

In [ ]:
res, model_datacube, flx_params = ifulmodel4.generate_residuals(init_params, return_datacube=True, linear_solve=True, vd_plots=True)

In [ ]:
ifulmodel4.generate_source_plots(np.array(lens_model_initparams + v_los_initparams + v_disp_initparams + list(flx_params)), 
                                 dpix=ifulmodel4.dpix)

In [ ]:
filename = f"s4_models/animation_init.gif"
gen_gif(
    ifulmodel4.obs_datacube,
    model_datacube,
    ifulmodel4.datacube_unc**2,
    ifulmodel4.datacube_mask,
    ifulmodel4.imset.wavelength,
    filename, overwrite=False
)
imp(filename=filename)

## Fitting only IFUL parameters (exclude lens model parameters)

In [ ]:
options = {"c1": 0.5, "c2": 0.3, "w": 0.9}
nwalkers = 240

def min_fnc(set_params):
    all_res = []
    for params in set_params:
        all_res += [ifulmodel4.generate_residuals(lens_model_initparams + list(params), linear_solve=True)]
    return all_res

# iful_lowerbounds = np.array([0, 0, 0, 1.430 * c, 1, 1.0] + list(np.zeros(len(ifulmodel4.init_bin_sourceflux))))
# iful_upperbounds = np.array([360, 1000, 10, 1.436 * c, 500, 3.0] + list(1e5+np.zeros(len(ifulmodel4.init_bin_sourceflux))))

iful_lowerbounds = np.array([0, 0, 0, 1.430 * c, 0., 5.0])
iful_upperbounds = np.array([360, 1000, 10, 1.436 * c, 300, 10.0])

all_init_fits = [v_los_initparams + v_disp_initparams + flx_initparams]
while len(all_init_fits) < nwalkers:
    all_init_fits_t = [
        np.random.uniform(80, 100),
        np.random.uniform(100, 400),
        np.random.uniform(3, 7),
        np.random.uniform(1.432, 1.434) * c,
        np.random.uniform(30, 80),
        np.random.uniform(6.0, 9.0),
    ]
    # all_init_fits_t += list(np.array(flx_initparams) * np.random.uniform(0.5, 1.5))
    all_init_fits_t = np.array(all_init_fits_t)
    if np.all(all_init_fits_t >= iful_lowerbounds) and np.all(
all_init_fits_t <= iful_upperbounds
    ):
        all_init_fits += [all_init_fits_t]
all_init_fits = np.array(all_init_fits)

optimizer = ps.single.GlobalBestPSO(
    n_particles=nwalkers,
    dimensions=len(all_init_fits[0]),
    options=options,
    bounds=[iful_lowerbounds, iful_upperbounds],
    init_pos=all_init_fits,
)

res_key = "_".join(iful_profiles)+"_ifulonly"
if res_key in previous_results:
    init_params = previous_results[res_key]
else:
    cost, pos = optimizer.optimize(min_fnc, iters=100, n_processes=threadCount)
    init_params = lens_model_initparams + list(pos)
    previous_results[res_key] = init_params
    with open(iful_pso_results_filename, "wb") as handle:
        pickle.dump(previous_results, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
def check_bounds_proximity(params, lower_bounds, upper_bounds, atol=1e-3, rtol=1e-5):
    params = np.asarray(params)
    lower_bounds = np.asarray(lower_bounds)
    upper_bounds = np.asarray(upper_bounds)
    lower_margin = lower_bounds + atol + rtol * np.abs(lower_bounds)
    upper_margin = upper_bounds - (atol + rtol * np.abs(upper_bounds))
    at_or_past_lower = params <= lower_margin
    at_or_past_upper = params >= upper_margin
    any_violation = at_or_past_lower | at_or_past_upper

    return {
        "lower_violation": at_or_past_lower,
        "upper_violation": at_or_past_upper,
        "any_violation": any_violation,
        "is_safe": not any_violation.any() 
    }
    
results = check_bounds_proximity(init_params[len(lens_model_initparams):], iful_lowerbounds, iful_upperbounds)
print("Violates Lower Bounds:", results['lower_violation'])
print("Violates Upper Bounds:", results['upper_violation'])
print("Any Violation Map:", results['any_violation'])

In [ ]:
res, model_datacube, flx_params = ifulmodel4.generate_residuals(init_params, return_datacube=True, linear_solve=True, vd_plots=True)

In [ ]:
ifulmodel4.generate_source_plots(np.array(list(init_params) + list(flx_params)))

In [ ]:
filename = f"s4_models/animation_ifulonly.gif"
gen_gif(
    ifulmodel4.obs_datacube,
    model_datacube,
    ifulmodel4.datacube_unc**2,
    ifulmodel4.datacube_mask,
    ifulmodel4.imset.wavelength,
    filename, overwrite=False
)
imp(filename=filename)

## Fitting all parameters (lensing and IFUL)

In [ ]:
options = {"c1": 0.5, "c2": 0.3, "w": 0.9}

def min_fnc(set_params, priors=[(6, 1.104, 0.025), (7, 1.588, 0.041)]):
    all_res = []
    for params in set_params:
        chi2_data = ifulmodel4.generate_residuals(params, linear_solve=True)
        chi2_prior = 0.0
        if priors is not None:
            for idx, mu, sigma in priors:
                chi2_prior += ((params[idx] - mu) / sigma) ** 2
        all_res.append(chi2_data + chi2_prior)
    return all_res
    
lensing_lower_bounds, lensing_upper_bounds = ifulmodel4.init_fitting_seq.likelihoodModule.param_limits
iful_lowerbounds = np.array(list(lensing_lower_bounds) + list(iful_lowerbounds))
iful_upperbounds = np.array(list(lensing_upper_bounds) + list(iful_upperbounds))

all_init_fits = [init_params]
while len(all_init_fits) < nwalkers:
    all_init_fits_t = np.array(init_params) * np.random.normal(1, 0.05, len(init_params))
    if np.all(all_init_fits_t >= iful_lowerbounds) and np.all(all_init_fits_t <= iful_upperbounds):
        all_init_fits += [all_init_fits_t]
all_init_fits = np.array(all_init_fits)

optimizer = ps.single.GlobalBestPSO(
    n_particles=nwalkers,
    dimensions=len(all_init_fits[0]),
    options=options,
    bounds=[iful_lowerbounds, iful_upperbounds],
    init_pos=all_init_fits,
)

res_key = "_".join(iful_profiles)+"_ifulall"
if res_key in previous_results:
    init_params = previous_results[res_key]
else:
    cost, pos = optimizer.optimize(min_fnc, iters=300, n_processes=threadCount)
    init_params = list(pos)
    previous_results[res_key] = init_params
    with open(iful_pso_results_filename, "wb") as handle:
        pickle.dump(previous_results, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
results = check_bounds_proximity(init_params, iful_lowerbounds, iful_upperbounds)
print("Violates Lower Bounds:", results['lower_violation'])
print("Violates Upper Bounds:", results['upper_violation'])
print("Any Violation Map:", results['any_violation'])

In [ ]:
res, model_datacube, flx_params = ifulmodel4.generate_residuals(init_params, return_datacube=True, linear_solve=True, vd_plots=True)

In [ ]:
ifulmodel4.generate_source_plots(np.array(list(init_params) + list(flx_params)))

In [ ]:
filename = f"s4_models/animation_ifulall.gif"
gen_gif(
    ifulmodel4.obs_datacube,
    model_datacube,
    ifulmodel4.datacube_unc**2,
    ifulmodel4.datacube_mask,
    ifulmodel4.imset.wavelength,
    filename, overwrite=False
)
imp(filename=filename)

## Run EMCEE

In [ ]:
def log_prob(params, priors=[(6, 1.104, 0.025), (7, 1.588, 0.041)]):
    if not (np.all(params >= iful_lowerbounds) and np.all(params <= iful_upperbounds)):
        return -np.inf
    
    chi2_data = ifulmodel4.generate_residuals(params, linear_solve=True)
    chi2_prior = 0.0
    if priors is not None:
        for idx, mu, sigma in priors:
            chi2_prior += ((params[idx] - mu) / sigma) ** 2
            
    return -0.5 * (chi2_data + chi2_prior)

In [ ]:
ndim = len(init_params)
mcmc_nwalkers = 5 * ndim
param_names = [f"param_{i}" for i in range(ndim)] # Define for printing

# --- Run Settings ---
run_comp_len = 100
max_iterations = 100
moves = [(emcee.moves.DEMove(), 0.8), (emcee.moves.DESnookerMove(), 0.2)] 

# --- File Management ---
model_dir = "s4_models"
os.makedirs(model_dir, exist_ok=True)

bf = f"{model_dir}/model_backup.hdf5"
first = not os.path.isfile(bf)
converged_b4 = os.path.isfile(f'{model_dir}/CONVERGED.txt')
convergence = False

if not os.path.isfile(f"{model_dir}/bandend_i.txt"):
    with open(f'{model_dir}/bandend_i.txt', "w") as f:
        f.write("0")
        
with open(f'{model_dir}/bandend_i.txt') as f:
    bandend_i = int(f.readlines()[-1])
    
# Attach the HDF5 backend
backend = emcee.backends.HDFBackend(bf, name="custom_mcmc_emcee")

# Add backend.iteration == 0 to catch cases where the file exists but is empty
if first or backend.iteration == 0: 
    backend.reset(mcmc_nwalkers, ndim)
    init_pos_mcmc = np.array(init_params) + 1e-4 * np.random.randn(mcmc_nwalkers, ndim)
    pos = np.clip(init_pos_mcmc, iful_lowerbounds + 1e-8, iful_upperbounds - 1e-8)
else:
    # Explicitly pull the last state from the HDF5 file to resume
    pos = backend.get_last_sample()

old_tau = np.inf

with Pool(threadCount) as pool:
    
    # Initialize the native emcee sampler
    sampler = emcee.EnsembleSampler(
        mcmc_nwalkers, ndim, log_prob, 
        pool=pool, backend=backend, moves=moves
    )

    while True:
        if not converged_b4:
            bandend_i += 1
            print(f"\n--- Starting run chunk {bandend_i} ---")

        if converged_b4 or convergence:
            os.system(f"touch {model_dir}/CONVERGED.txt")
            print("Chains have converged! Exiting loop.")
            break

        # Execute MCMC chunk. It auto-appends to the HDF5 backend.
        sampler.run_mcmc(pos, run_comp_len, progress=True)
        
        # Ensure pos is None for subsequent loops so it continues from the backend
        pos = None 

        with open(f'{model_dir}/bandend_i.txt', "w") as f:
            f.write(f"{bandend_i}")

        total_steps = sampler.iteration
        print(f"Total accumulated iterations in backend: {total_steps}")

        # Calculate autocorrelation time (tol=0 suppresses warnings on short chains)
        tau = sampler.get_autocorr_time(tol=0)
        max_tau = np.max(tau)
        
        # 1. Check if chain is long enough (20x the autocorrelation time)
        converged = np.all(tau * 20 < total_steps)
        
        # 2. Check if tau estimate has stabilized (< 5% change from last chunk)
        if not np.isinf(old_tau).any():
            tau_diff = np.abs(old_tau - tau) / tau
            converged &= np.all(tau_diff < 0.05)
        else:
            # Cannot check stability on the first pass
            converged = False
            
        old_tau = tau

        print(f"Max autocorrelation time (tau): {max_tau:.2f}")
        print(f"Current iterations: {total_steps} (Target for convergence: > {20 * max_tau:.2f})")
        print(f"CONVERGENCE : {converged}")
        
        # Dynamically discard burn-in based on tau
        burnin = int(2 * max_tau) if max_tau > 0 else 0
        if burnin >= total_steps:
            burnin = total_steps // 2

        print(f"Discarding {burnin} steps as burn-in for summary statistics...")
        
        try:
            flat_chain = sampler.get_chain(discard=burnin, flat=True)
            medians = np.median(flat_chain, axis=0)
            stds = np.std(flat_chain, axis=0)

            for i, (p, med, mstd) in enumerate(zip(param_names, medians, stds)):
                conv_flag = (tau[i] * 20 < total_steps)
                print(f"{p:<25}: {med:>15.5f} +- {mstd:>15.5f}    tau: {tau[i]:>6.1f}   conv: {conv_flag}")
        except ValueError:
            print("Chain still too short to compute reliable summary statistics.")

        first = False

        if converged:
            convergence = True
        elif bandend_i >= max_iterations:
            os.system(f"touch {model_dir}/fail_to_converge.txt")
            print("Reached maximum chunks without converging. Exiting loop.")
            break

In [ ]:
bf = "s4_models/model_backup.hdf5"
reader = emcee.backends.HDFBackend(bf, name="custom_mcmc_emcee")

# 2. Get the full raw chain to check autocorrelation time
# Shape is (n_steps, n_walkers, n_dim)
raw_chain = reader.get_chain()
ndim = raw_chain.shape[2]
param_names = [f"param_{i}" for i in range(ndim)]

# Calculate tau to determine a good burn-in discard
try:
    tau = reader.get_autocorr_time(tol=0)
    burnin = int(2 * np.max(tau))
    print(f"Estimated burn-in (2x max tau): {burnin} steps")
except emcee.autocorr.AutocorrError:
    print("Chain is too short for reliable tau estimation. Defaulting to 0 burn-in.")
    burnin = 0

# 3. Generate Trace Plots
fig, axes = plt.subplots(ndim, 1, figsize=(10, 2.5 * ndim), sharex=True)

# Note: We plot the raw chain here so you can visually see the burn-in phase.
# If you only want to see the converged part, change this to:
# plot_chain = reader.get_chain(discard=burnin)
plot_chain = raw_chain 

for i in range(ndim):
    ax = axes[i]
    # Plot all walkers for parameter i
    # alpha=0.3 makes overlapping walkers easier to read
    ax.plot(plot_chain[:, :, i], "k", alpha=0.3)
    
    ax.set_xlim(0, len(plot_chain))
    ax.set_ylabel(param_names[i], fontsize=14)
    
    # Draw a vertical red line to show where burn-in was discarded
    if burnin > 0 and burnin < len(plot_chain):
        ax.axvline(burnin, color="red", linestyle="--", lw=2, label="Burn-in cut" if i==0 else "")

axes[-1].set_xlabel("Step Number", fontsize=14)
if burnin > 0:
    fig.legend(loc="upper right")
    
plt.tight_layout()
plt.show()

In [ ]:
flat_samples = reader.get_chain(discard=burnin, flat=True)

fig = corner.corner(
    flat_samples, 
    labels=param_names,
    quantiles=[0.16, 0.5, 0.84], # 1-sigma percentiles
    show_titles=True, 
    title_kwargs={"fontsize": 12}
)
plt.show()